In [1]:
import numpy as np 
import pandas as pd 

In [2]:
# SYSTEM SPECIFICATIONS

import platform
import psutil
import cpuinfo # the package is called 'py-cpuinfo'

print("\n### System Specifications ###")
print("System:", platform.system())
print("Architecture:", cpuinfo.get_cpu_info()["arch"])
print("CPU:", cpuinfo.get_cpu_info()["brand_raw"])
print("CPU Cores:", psutil.cpu_count(logical=True)) 
print("Memory:", round(psutil.virtual_memory().total / (1024 ** 3), 2), "Gigabytes")
print("Python", platform.python_version())


### System Specifications ###
System: Windows
Architecture: X86_64
CPU: AMD Ryzen 7 7800X3D 8-Core Processor
CPU Cores: 16
Memory: 31.12 Gigabytes
Python 3.12.4


# Data Loading

In [3]:
# define the features that will be used
usecols = ['datadate', 'GVKEY', 'LPERMNO', 'LPERMCO', 
    'conm', 'conml', 'gind', 'ggroup', 
    'curcdd', 'adrrc', 'ajexdi', 
    'cshoc', 'cshtrd', 'prccd', 
    'trfd', 'exchg', 'tpci', 
    'dlrsn', 'gsector']

# define data types explicitly
dtypes = {'datadate': np.string_, 'GVKEY': np.float32, 'LPERMNO': np.float32, 'LPERMCO': np.float32, 
    'conm': np.string_, 'conml': np.string_, 'gind': np.string_, 'ggroup': np.string_,
    'curcdd': np.string_, 'adrrc': np.float32, 'ajexdi': np.float32, 
    'cshoc': np.float32, 'cshtrd': np.float32, 'prccd': np.float32, 
    'trfd': np.float32, 'exchg': np.float32, 'tpci': np.string_, 
    'dlrsn': np.string_, 'gsector': np.string_}

# read the data from a .csv file
data = pd.read_csv("./DATA/CRSP.csv", low_memory=True, header=0, dtype=dtypes, usecols=usecols)

In [4]:
# print unique values for features of interest
features_of_interest = ['curcdd', 'gsector']
print('unique values:')
for feature in features_of_interest:
    print(f"{feature}:", pd.unique(data[feature]))

# print percentage of N/A values for features of interest
features_of_interest = ['curcdd', 'GVKEY', 'LPERMNO', 'LPERMCO']
print('\nN/A percentage:')
for feature in features_of_interest:
    print(f"{feature}: {np.round(np.sum(data[feature].isna()) / data.shape[0] * 100, 3)}%")

# notable observations:
#   1. USD is the only currency
#   2. there are 11 sectors: ['20', '45', '35', '25', '40', '55', '15', '10', '50', '30', '60']
#   3. the N/A percentage of the currency code ('curcdd') is 0.007%
#   4. the N/A percentage of the company key ('GVKEY') is 0.0%
#   5. the N/A percentage of the security number ('LPERMNO') is 0.0%
#   6. the N/A percentage of the company number ('LPERMCO') is 0.0%

unique values:
curcdd: ['USD' nan]
gsector: ['20' '45' '35' '25' '40' '55' '15' nan '10' '50' '30' '60']

N/A percentage:
curcdd: 0.007%
GVKEY: 0.0%
LPERMNO: 0.0%
LPERMCO: 0.0%


# Data Preparation

In [5]:
# convert 'datadate' to datetime format
data['datadate'] = pd.to_datetime(data['datadate'])

# 'cshoc' is not availabe prior to april 1998, cut the data there
cutoff_date = pd.to_datetime('1998-03-31')
data = data[data['datadate'] >= cutoff_date]

In [6]:
# SELECTING COMMON STOCKS
# select stocks with type 'tpci' = 0, F
#   0: common
#   F: depositary receipt

data = data[(data["tpci"] == "0") | (data["tpci"] == "F")]


In [7]:
# FILTERING BY STOCK EXCHANGE
# exchange codes:
#   "Broker": 127
#   "Fund Manager": 283
#   "Fund Managers": 157
#   "OTC": 229
#   "OTC Bulletin Board": 13
#   "Other-OTC": 19
#   "Subsidiary/Private": 0
#   "Unlisted Evaluated Equity": 20
#   "Unlisted Securities Market": 269
#   "Non-traded Company or Security": 1
#   "Stock Connect": 341, 348, 342, 347


exchange_codes = [127, 283, 157, 229, 13, 19, 0, 20, 269, 1, 341, 348, 342, 347]

for code in exchange_codes:
    data = data[data["exchg"] != code]


# exchanges left over:
#   11: New York Stock Exchange 
#   14: Nasdaq Stock Market 
#   12: NYSE American
#   17: NYSE Arca 
#   18: Nasdaq PHLX 
#   16: NYSE Chicago
#   21: Cboe BZX Exchange 
#   15: Nasdaq BX

print(pd.unique(data["exchg"]))

data = data.drop(columns="exchg", axis=1)

[11. 14. 12. 17. 18. 16. 21. 15.]


In [8]:
# exclude REITs:
#   gind = 402040

unique_ids_before = len(pd.unique(data["LPERMNO"]))

data = data[data["gind"] != "402040"]

unique_ids_after = len(pd.unique(data["LPERMNO"]))

print("Excluded", unique_ids_before - unique_ids_after, "securities.")

data = data.drop(columns="gind", axis=1)

Excluded 49 securities.


In [9]:
# create boolean mask for banks
is_bank = (data['ggroup'] == "4010")

# copy bank rows to separate dataframe
banks = data[is_bank].copy()

# remove banks from original dataframe (so they can be added back later)
data = data[~is_bank]

print(f"Number of securities identified as coming from a bank: {len(pd.unique(banks["LPERMNO"]))}")

data = data.drop(columns="ggroup", axis=1)

Number of securities identified as coming from a bank: 1267


In [10]:
# EXCLUDE FUNDS AND TRUSTS:
#   where 'conm', 'conml' includes “_fund”, “_trust”, “venture_capital_trust”, “_vct”, or “_reit”

patterns = [' fund', ' trust', 'venture capital trust', ' vct', ' reit']
full_pattern = '|'.join(patterns)

unique_ids_before = len(pd.unique(data["LPERMNO"]))

data = data[~data['conm'].str.contains(full_pattern, case=False, na=False)]
data = data[~data['conml'].str.contains(full_pattern, case=False, na=False)]

unique_ids_after = len(pd.unique(data["LPERMNO"]))

print("Excluded", unique_ids_before - unique_ids_after, "securities.")

data = data.drop(columns="conm", axis=1)
data = data.drop(columns="conml", axis=1)

Excluded 178 securities.


In [11]:
# add the banks back
data = pd.concat([data, banks], ignore_index=True)

In [12]:
# ADR RATIO CORRECTION
# if the security is a depositary receipt, do CSHOC = CSHOC / ADRRC
mask = (data['tpci'] == 'F')
data.loc[mask, 'cshoc'] /= data.loc[mask, 'adrrc']

data = data.drop(columns=["adrrc", "tpci"], axis=1)

In [ ]:
# Deal with potentially re-used permanent security numbers:

# sort to detect changes chronologically
data = data.sort_values(['LPERMNO', 'datadate'])

# within each LPERMNO, check where GVKEY changes
# GVKEY_change is 1 for the first firm, 2 after a GVKEY switch
data['GVKEY_change'] = (
    data.groupby('LPERMNO')['GVKEY']
    .transform(lambda s: s.ne(s.shift()).cumsum())
)

# make LPERMNO unique by adding a suffix:
# add the GVKEY_change number at the end
data['LPERMNO_adj'] = data['LPERMNO'].astype(str)
mask = data['GVKEY_change'] > 1

data.loc[mask, 'LPERMNO_adj'] = data.loc[mask, 'LPERMNO'].astype(str) + data.loc[mask, 'GVKEY_change'].astype(str)

# print number of LPERMNOs that were re-used
reused = data.groupby('LPERMNO')['GVKEY_change'].max()
n_reused = (reused > 1).sum()
print(f"Found {n_reused} reused numbers.")


# drop columns that are no longer needed
data = data.drop(columns=['GVKEY', 'GVKEY_change', 'LPERMNO'], axis=1)


Found 387 reused numbers.


In [14]:
# PIVOT
# from long to wide format

# get all columns except 'datadate'
other_columns = [col for col in data.columns if col not in ['datadate']]

# pivot the dataframe around 'datadate' and 'LPERMNO'
wide = data.pivot(
    index='datadate',
    columns='LPERMNO_adj',
    values=other_columns
)

# sort the column index
wide = wide.sort_index(axis=1, level=[0, 1])

# only use columns that are actually needed from here onward to save memory
usecols = ['ajexdi', 'LPERMCO', 'cshoc', 'cshtrd', 'dlrsn', 'gsector', 'prccd', 'trfd'] 
wide = wide[usecols]

del data # free up memory

# prevent fragmentation
wide = wide.copy()

In [15]:
# drop stocks whose adjustment factor ever becomes 0 or negative
has_zero = (wide['ajexdi'] <= 1e-9).any(axis=0)
tickers_to_keep = has_zero[~has_zero].index
wide = wide.loc[:, (slice(None), tickers_to_keep)]

print("Number of stocks remaining:", len(tickers_to_keep))

Number of stocks remaining: 14099


In [16]:
# CALCULATE RETURNS

wide_lagged = wide.shift(periods=1, axis=0, fill_value=np.nan)

prc = wide.xs('prccd', level=0, axis=1)
ajexdi = wide.xs('ajexdi', level=0, axis=1)
trfd = wide.xs('trfd', level=0, axis=1)

prc_lagged = wide_lagged.xs('prccd', level=0, axis=1)
ajexdi_lagged = wide_lagged.xs('ajexdi', level=0, axis=1)
trfd_lagged = wide_lagged.xs('trfd', level=0, axis=1)

ret = (prc / ajexdi * trfd) / (prc_lagged / ajexdi_lagged * trfd_lagged) - 1

# free up space
del wide_lagged
del prc
del ajexdi
del trfd
del prc_lagged
del ajexdi_lagged
del trfd_lagged

ret.columns = pd.MultiIndex.from_product([['ret'], ret.columns])
wide = pd.concat([wide, ret], axis=1)

# free up space
del ret

# drop columns we no longer need to save memory
wide = wide.sort_index(axis=1, level=[0, 1])
wide = wide.drop(["ajexdi", "trfd"], axis=1)

# prevent fragmenting of the dataframe
wide = wide.copy()

In [17]:
# CALCULATE MARKET CAPITALIZATION 

prc = wide.xs('prccd', level=0, axis=1)
cshoc = wide.xs('cshoc', level=0, axis=1)
sz = prc * cshoc
sz.columns = pd.MultiIndex.from_product([['sz'], sz.columns])
wide = pd.concat([wide, sz], axis=1)

# free up space
del prc
del cshoc
del sz

# drop columns we no longer need to save memory
wide = wide.sort_index(axis=1, level=[0, 1])
wide = wide.drop("cshoc", axis=1)

# prevent fragmenting of the dataframe
wide = wide.copy()


In [18]:
# delete the first month of data for all stocks (first 21 trading days) 

for number in wide.columns.get_level_values('LPERMNO_adj').unique():
    ret_series = wide[('ret', number)]
    
    # find the first non-nan index
    first_valid_idx = ret_series.first_valid_index()

    if first_valid_idx is not None:
        # calculate the range (first 21 observations)
        first_valid_pos = wide.index.get_loc(first_valid_idx)
        end_pos = min(first_valid_pos + 21, len(wide))
        idx_range = wide.index[first_valid_pos:end_pos]

        # set values in range to N/A
        wide.loc[idx_range, ('ret', number)] = np.nan


# prevent fragmenting of the dataframe
wide = wide.copy()

In [19]:
# DEALING WITH MISSING DATA

wide = wide.sort_index(axis=1, level=[0, 1])
pd.set_option('future.no_silent_downcasting', True)


def fill_and_count(df, method="ffill", desc=""):
    """
    Arguments:
    - df: view of dataframe whose missing values are to be filled
    - method: forward or backward filling, named after the pandas functions (either 'ffill' or 'fillna')
    - desc: description of the data for the print out

    Returns: Filled dataframe.

    """

    inside_total = 0
    inside_nans = 0

    # count the number of N/A values and the total number of values in the "inside" area of the data
    for col in df.columns:
        s = df[col]
        first = s.first_valid_index()
        last = s.last_valid_index()
        if first is not None:
            inside = s.loc[first:last]
            inside_total += len(inside)
            inside_nans += inside.isna().sum()

    # fill the data according to the specified method
    # and print how many values were filled this way
    if method == "ffill":
        df_filled = df.ffill(axis=0, limit_area="inside")
        print(f"{desc}: {inside_nans} values ({inside_nans / inside_total:.3%}) were forward filled.")
    elif method == "fillna":
        # fillna() lacks the limit_area parameter, so must find other solution:
        mask = (df.ffill(axis=0).notna() & df.bfill(axis=0).notna())
        df_filled = df.fillna(value=0.0).where(mask, df)
        print(f"{desc}: {inside_nans} values ({inside_nans / inside_total:.3%}) were filled with 0.0.")
    else:
        print("Method not specified correctly!")
        return 0


    return df_filled


# fill missing returns with 0.0
wide["ret"] = fill_and_count(df=wide["ret"], method="fillna", desc="Return")

# forward fill missing market capitalizations.
wide["sz"] = fill_and_count(df=wide["sz"], method="ffill", desc="Market Capitalization")

# forward fill missing sector codes
wide["gsector"] = fill_and_count(df=wide["gsector"], method="ffill", desc="Sector Codes")


# prevent fragmenting of the dataframe
wide = wide.copy()

Return: 324025 values (1.336%) were filled with 0.0.
Market Capitalization: 403734 values (1.246%) were forward filled.
Sector Codes: 405171 values (1.252%) were forward filled.


In [20]:
# SOME N/A FILTERS

# 1. drop days where all stocks have a return of 0 (i.e. keep only rows where any stock has a non-zero return)
ret_cols = wide.xs('ret', level=0, axis=1)
mask = (ret_cols != 0) & ret_cols.notna()
wide = wide[mask.any(axis=1)]


# 2. drop stocks who have too many returns equal to 0

wide = wide.sort_index(axis=1, level=[0, 1])
numbers = wide.columns.get_level_values(1)

# find stocks to keep (with less than or equal to x% zeros)
numbers_to_keep = []
x = 20

for number in numbers:
    ret_col = wide[('ret', number)]
    
    non_nan_count = ret_col.notna().sum()
    if non_nan_count == 0:
        continue
    zero_count = (ret_col == 0).sum()
    zero_percentage = (zero_count / non_nan_count) * 100
    
    # keep stock if zero percentage is <= threshold
    if zero_percentage <= x:
        numbers_to_keep.append(number)

# filter dataframe to keep only columns with selected stocks
wide = wide.loc[:, wide.columns.get_level_values(1).isin(numbers_to_keep)]

n = len(wide.columns.get_level_values(0).unique())

original = int(len(numbers) / n)
dropped = int((len(numbers) - len(numbers_to_keep)) / n)
remaining = int(len(numbers_to_keep) / n)

print("Number of stocks originally:", original)
print("Number of stocks dropped:", dropped, f"({dropped/original:.2%})")
print("Number of stocks remaining:", remaining)


# 3. (AGAIN) drop days where all stocks have a return of 0
# this is done again because the previous step (2.) may have introduced more dates where this is the case
ret_cols = wide.xs('ret', level=0, axis=1)
mask = (ret_cols != 0) & ret_cols.notna()
wide = wide[mask.any(axis=1)]

# free up space
del ret_cols


Number of stocks originally: 14099
Number of stocks dropped: 6112 (43.35%)
Number of stocks remaining: 7987


In [21]:
# CALCULATE DOLLAR VOLUME

prc = wide.xs('prccd', level=0, axis=1)
cshtrd = wide.xs('cshtrd', level=0, axis=1)
dvol = prc * cshtrd
dvol.columns = pd.MultiIndex.from_product([['dvol'], dvol.columns])
wide = pd.concat([wide, dvol], axis=1)

# free up space
del prc
del cshtrd
del dvol

# drop columns that are no longer needed to save memory
wide = wide.sort_index(axis=1, level=[0, 1])
wide = wide.drop("cshtrd", axis=1)

# prevent fragmenting of the dataframe
wide = wide.copy()

In [22]:
# delete the remaining return history for any stock whose market capitalization drops below 1 million, or whose share price drops below 0.01 USD

for number in wide.columns.get_level_values('LPERMNO_adj'):
    #create condition
    condition = (wide[('sz', number)] < 1000000) | (wide[('prccd', number)] < 0.01)
    
    if condition.any():
        first_breach = condition.idxmax() #find the first occurrence

        wide.loc[first_breach:, ('ret', number)] = np.nan
        wide.loc[first_breach:, ('sz', number)] = np.nan

In [23]:
# drop stocks who now have less than 18 months of data (we need at least a 1-year history for the optimization, and another 3 months for the holding period)

numbers_to_keep = []

for number in wide.columns.get_level_values('LPERMNO_adj').unique():
    ret_series = wide[('ret', number)]

    # note that because of the filling we did earlier, the non-N/As must necessarily be consecutive
    non_nan_count = ret_series.notna().sum()
    
    if non_nan_count >= 378: # 378 = 21 days * 18 months
        numbers_to_keep.append(number)

wide = wide.loc[:, (slice(None), numbers_to_keep)]

print("Number of stocks remaining:", len(numbers_to_keep))

Number of stocks remaining: 7175


In [24]:
# drop stocks with unreliable data, i.e. when there is a gap of more than 1 month (21 trading days) with no price update
numbers_to_remove = []

for number in wide.columns.get_level_values('LPERMNO_adj').unique():
    ret_series = wide[('ret', number)]
    is_zero = (ret_series == 0.0)
    
    #find consecutive sequences of zeros and group them using cumsum
    groups = (is_zero != is_zero.shift()).cumsum()
    consecutive_counts = is_zero.groupby(groups).sum()

    if (consecutive_counts > 21).any():
        numbers_to_remove.append(number)

numbers_to_keep = [number for number in wide.columns.get_level_values('LPERMNO_adj').unique() if number not in numbers_to_remove]

wide = wide.loc[:, (slice(None), numbers_to_keep)]

print("Number of stocks remaining:", len(numbers_to_keep))

Number of stocks remaining: 7113


In [25]:
# company/number problem: we have multiple unique shares from the same company

# 1. make a dictionary of duplicates:
number_to_company = {}

for number in wide.columns.get_level_values('LPERMNO_adj').unique(): 
    # get all company names for this number (across all dates)
    company_values = wide[('LPERMCO', number)].dropna().unique()

    # map each company to this number
    for company in company_values:
        if company not in number_to_company:
            number_to_company[company] = []
        number_to_company[company].append(number)

# convert to series with unique numbers per company
grouped = pd.Series({k: list(set(v)) for k, v in number_to_company.items()})
grouped = grouped[grouped.apply(len) > 1] # only include companies who have multiple numbers associated with them
duplicates_dict = grouped.to_dict()


# 2. filter by average trading volume
numbers_to_drop = []

# for each company in the dictionary
for company_name, numbers in duplicates_dict.items():

    # calculate average trading volume for each number
    avg_dvol = {}
    for number in numbers:
        avg_dvol[number] = wide[('dvol', number)].mean(skipna=True)
   
    # sort by average volume descending (highest first)
    sorted_numbers = sorted(numbers, key=lambda n: avg_dvol[n], reverse=True)

    kept = [sorted_numbers[0]]  # always keep the security with the highest average dollar volume

    for number in sorted_numbers[1:]:
        ret_current = wide[('ret', number)].dropna().index
        
        # check if this security overlaps with any kept security
        overlaps = False
        for kept_number in kept:
            ret_kept = wide[('ret', kept_number)].dropna().index

            if len(ret_current.intersection(ret_kept)) > 0: # compare the indices, if they overlap, then the securities existed simultaneously
                overlaps = True
                break
        
        if overlaps:
            # if there is overlap with any kept security of the company, drop it
            if number in wide.columns.get_level_values(1):
                numbers_to_drop.append(number)
            
        else:
            # if there is no overlap with any kept security of the company, keep it
            kept.append(number)

wide = wide.drop(columns=numbers_to_drop, level=1)

remaining = wide["dvol"].shape[1]
print("Number of duplicates dropped:", len(numbers_to_drop))
print("Number of stocks remaining:", remaining)


Number of duplicates dropped: 98
Number of stocks remaining: 7015


In [26]:
# set final returns and market caps if there is a delisting

for ticker in wide.columns.get_level_values('LPERMNO_adj').unique():
    last_valid_idx = wide[('ret', ticker)].last_valid_index()
    
    if last_valid_idx is not None:
        idx_position = wide.index.get_loc(last_valid_idx)
        
        # check if there exists a next day in the dataframe
        if idx_position < len(wide.index) - 1:
            # check if delisting reason is liquidation or bankruptcy
            if (wide.loc[last_valid_idx, ('dlrsn', ticker)] == 2.) | (wide.loc[last_valid_idx, ('dlrsn', ticker)] == 3.):
                next_day = wide.index[idx_position + 1]

                # set return to -30% on the next day
                wide.loc[next_day, ('ret', ticker)] = -0.3
                
                # set market capitalization to (1 - 30%) * previous day's market capitalization
                prev_sz = wide.loc[last_valid_idx, ('sz', ticker)]
                wide.loc[next_day, ('sz', ticker)] = 0.7 * prev_sz

In [27]:
# DEALING WITH OUTLIERS:

# order the index 
wide = wide.sort_index(axis=1, level=[0, 1])

std_cutoff = 0.5 # if any stock has a lifetime standard deviation of returns above 50%, drop it

stds_before = wide["ret"].std()
outlier_stocks = stds_before[stds_before > std_cutoff].index.tolist()

print("Number of stocks removed:", len(outlier_stocks),"\n")
for s in outlier_stocks:
    print(f"Stock PERMNO_adj: {s},  std = {stds_before[s]:.2f}")

wide = wide.drop(columns=outlier_stocks, level=1)

Number of stocks removed: 2 

Stock PERMNO_adj: 14610.0,  std = 0.55
Stock PERMNO_adj: 90724.0,  std = 32.72


In [28]:
# DEALING WITH OUTLIERS 2:

# order the index 
wide = wide.sort_index(axis=1, level=[0, 1])

ret_cutoff = 5 # if any stock ever has a return that exceeds 500% in one day, exclude it 

maxs_before = wide["ret"].max()
outlier_stocks = maxs_before[maxs_before > ret_cutoff].index.tolist()

print("Number of stocks removed:", len(outlier_stocks),"\n")
for s in outlier_stocks:
    print(f"Stock PERMNO_adj: {s},  max = {maxs_before[s]:.2f}")

wide = wide.drop(columns=outlier_stocks, level=1)

Number of stocks removed: 11 

Stock PERMNO_adj: 20636.0,  max = 6.48
Stock PERMNO_adj: 50606.0,  max = 25.88
Stock PERMNO_adj: 84302.0,  max = 28.31
Stock PERMNO_adj: 86233.0,  max = 11.32
Stock PERMNO_adj: 89169.0,  max = 12.14
Stock PERMNO_adj: 90298.0,  max = 10.34
Stock PERMNO_adj: 90700.0,  max = 14.85
Stock PERMNO_adj: 91845.0,  max = 13.71
Stock PERMNO_adj: 91973.0,  max = 6.77
Stock PERMNO_adj: 92345.0,  max = 8.75
Stock PERMNO_adj: 92520.0,  max = 9.51


In [29]:
# (AGAIN) drop days where all stocks have a return of 0
# this is done again because the previous filters may have introduced more dates where this is the case
ret_cols = wide.xs('ret', level=0, axis=1)
mask = (ret_cols != 0) & ret_cols.notna()
wide = wide[mask.any(axis=1)]

In [30]:
# order the index such that the identification numbers are in the same order for all the datasets saved
wide = wide.sort_index(axis=1, level=[0, 1])

# make sure that the columns match
assert np.all(np.sum(wide["ret"].columns == wide["sz"].columns))

# save the return and market capitalization data to .csv files for later use
wide["ret"].to_csv("./DATA/ret_full.csv")
wide["sz"].to_csv("./DATA/sz_full.csv")


In [31]:
# separately save the data grouped by sectors for the robustness test

ret = wide.xs('ret', level=0, axis=1)
sz = wide.xs('sz', level=0, axis=1)
gsector = wide.xs('gsector', level=0, axis=1)

# for each unique sector, mask 'ret' and 'sz' to only include returns and market capitalizations where the stock is actually in that sector
for sector in gsector.stack().unique():
    if pd.notna(sector):
        # mask: True where a stock is in this sector at this time
        mask = (gsector == sector)

        # apply the sector mask
        ret_subset = ret.where(mask)
        sz_subset = sz.where(mask)

        # find columns that have at least one valid value in both subsets
        valid_cols = ret_subset.columns[
            ret_subset.notna().any() & sz_subset.notna().any()
        ]

        # apply the same column selection to both (this is important!)
        ret_subset = ret_subset[valid_cols]
        sz_subset = sz_subset[valid_cols]

        # make sure that the columns match
        assert np.all(np.sum(ret_subset.columns == sz_subset.columns))

        # save the data to .csv files for later use
        ret_subset.to_csv(f'./DATA/ret_sector{sector}.csv')
        sz_subset.to_csv(f'./DATA/sz_sector{sector}.csv')